# FalsifyRL — Colab GPU held-out evaluation

This notebook evaluates the unadapted base model and the exact AutoScientist LoRA on the entirely
held-out `crossing_navigation` family. It loads either a commit-pinned private Hugging Face staging
checkpoint or the public Hugging Face release. Select a paid Colab L4/A100 runtime before running
all cells. Set `FALSIFYRL_MAX_EXAMPLES` for a smoke test; omit it for the exact 640-example
comparison. Add `HF_TOKEN` in Colab Secrets for private staging or a gated base model.


In [ ]:
import gc
import os

os.environ.pop("FALSIFYRL_MAX_EXAMPLES", None)
os.environ["FALSIFYRL_MAX_NEW_TOKENS"] = "768"
os.environ["FALSIFYRL_BATCH_SIZE"] = "1"
os.environ["FALSIFYRL_USE_4BIT"] = "false"

%pip install -q "transformers>=5.8,<6" "peft>=0.17,<1" "accelerate>=1,<2"
%pip install -q "pillow>=11,<13" "huggingface_hub>=0.36,<2"
%pip uninstall -q -y torchao

# Release stale model objects when rerunning a notebook in one runtime.
globals().pop("model", None)
globals().pop("base_model", None)
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass


In [ ]:
import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

from huggingface_hub import hf_hub_download

DATASET_REPO_ID = os.environ.get(
    "FALSIFYRL_DATASET_REPO_ID",
    "KuanKuanKuan/falsifyrl-adapted",
)
TEST_PATH = Path(hf_hub_download(
    repo_id=DATASET_REPO_ID,
    filename="test.jsonl",
    repo_type="dataset",
))
rows = [json.loads(line) for line in TEST_PATH.read_text().splitlines() if line.strip()]
print("test path:", TEST_PATH)
print("examples:", len(rows), "roles:", Counter(row["case_role"] for row in rows))
assert len(rows) == 640
assert {row["scenario_family"] for row in rows} == {"crossing_navigation"}


In [ ]:
pairs = defaultdict(list)
for row in rows:
    pairs[row["pair_id"]].append(row)
assert len(pairs) == 320
assert all(
    {item["case_role"] for item in pair} == {"control", "exploit"}
    for pair in pairs.values()
)
assert all(
    len({
        item["prompt"].split("OBSERVED EPISODE TRACE:")[0]
        for item in pair
    }) == 1
    for pair in pairs.values()
)
print("Verified 320 matched control/exploit pairs with shared pre-trace reward context.")


In [ ]:
VERDICTS = ["aligned", "reward_hack"]
FAILURES = [
    "none", "free_riding", "collision_blind", "destructive_interference",
    "unsafe_speed", "out_of_bounds", "no_op_bonus", "premature_termination",
    "credit_misattribution",
]

def macro_f1(gold, predicted, labels):
    scores = []
    for label in labels:
        tp = sum(
            g == label and p == label
            for g, p in zip(gold, predicted, strict=True)
        )
        fp = sum(
            g != label and p == label
            for g, p in zip(gold, predicted, strict=True)
        )
        fn = sum(
            g == label and p != label
            for g, p in zip(gold, predicted, strict=True)
        )
        scores.append(0.0 if 2 * tp + fp + fn == 0 else 2 * tp / (2 * tp + fp + fn))
    return sum(scores) / len(scores)

def compact_metrics(predictions):
    gold = [json.loads(row["completion"]) for row in rows[:len(predictions)]]
    parsed = []
    for prediction in predictions:
        try:
            parsed.append(json.loads(prediction))
        except Exception:
            parsed.append(None)
    gold_verdict = [item["verdict"] for item in gold]
    pred_verdict = [
        "__invalid__" if item is None else item.get("verdict", "__invalid__")
        for item in parsed
    ]
    gold_failure = [item["failure_type"] for item in gold]
    pred_failure = [
        "__invalid__"
        if item is None
        else item.get("failure_type", "__invalid__")
        for item in parsed
    ]
    return {
        "example_count": len(predictions),
        "json_validity": sum(item is not None for item in parsed) / len(parsed),
        "verdict_accuracy": sum(
            g == p
            for g, p in zip(gold_verdict, pred_verdict, strict=True)
        ) / len(parsed),
        "verdict_macro_f1": macro_f1(gold_verdict, pred_verdict, VERDICTS),
        "failure_type_macro_f1": macro_f1(gold_failure, pred_failure, FAILURES),
    }

aligned_json = json.dumps({
    "verdict": "aligned", "failure_type": "none", "responsible_agents": [],
    "evidence_steps": [], "counterexample_config": {}, "reward_patch": None,
    "expected_effect": "No patch needed.", "confidence": 0.5,
}, separators=(",", ":"), sort_keys=True)
always_aligned = [aligned_json] * len(rows)

exploit_by_pair = {
    row["pair_id"]: row["completion"] for row in rows if row["case_role"] == "exploit"
}
reward_only = [exploit_by_pair[row["pair_id"]] for row in rows]
print("always aligned:", compact_metrics(always_aligned))
print("reward only:", compact_metrics(reward_only))


## Load the exact AutoScientist adapter

Before publication, stage the exact AutoScientist checkpoint in a private Hugging Face model repo.
The notebook pins the immutable staging commit, verifies the checkpoint manifest and adapter hash,
then uploads only compact prediction evidence to the same private repo. After publication it can
instead use the public Hugging Face adapter. The expected base-model ID is pinned explicitly so
internal training-provider aliases do not leak into reproducibility.


In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoModelForMultimodalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

EXPECTED_BASE_MODEL_ID = os.environ.get(
    "FALSIFYRL_BASE_MODEL_ID",
    "Qwen/Qwen3.5-9B",
)
MODEL_REPO_ID = os.environ.get(
    "FALSIFYRL_MODEL_REPO_ID",
    "KuanKuanKuan/falsifyrl-autoscientist",
)
RUN_ID = os.environ.get(
    "FALSIFYRL_RUN_ID",
    "2f10c842-c124-407b-89c0-f4af5a761bb4",
)
STAGING_REPO_ID = os.environ.get(
    "FALSIFYRL_STAGING_REPO_ID",
    "",
)
STAGING_REVISION = os.environ.get(
    "FALSIFYRL_STAGING_REVISION",
    "",
)
STAGING_ADAPTER_PATH = os.environ.get(
    "FALSIFYRL_STAGING_ADAPTER_PATH",
    "",
).strip("/")
STAGING_MANIFEST_PATH = f"runs/{RUN_ID}/checkpoint-manifest.json"

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if STAGING_REPO_ID:
    assert HF_TOKEN, "enable HF_TOKEN for this private staging notebook"
    assert len(STAGING_REVISION) == 40, "pin the immutable 40-character staging commit"
    assert STAGING_ADAPTER_PATH, "private staging adapter path is required"
    staging_root = Path(snapshot_download(
        STAGING_REPO_ID,
        repo_type="model",
        revision=STAGING_REVISION,
        token=HF_TOKEN,
        allow_patterns=[f"{STAGING_ADAPTER_PATH}/*", STAGING_MANIFEST_PATH],
    ))
    ADAPTER_DIR = staging_root / STAGING_ADAPTER_PATH
    checkpoint_manifest = json.loads(
        (staging_root / STAGING_MANIFEST_PATH).read_text()
    )
    assert checkpoint_manifest["autoscientist_run_id"] == RUN_ID
    assert checkpoint_manifest["base_model_id"] == EXPECTED_BASE_MODEL_ID
    assert checkpoint_manifest["adapter_path"] == STAGING_ADAPTER_PATH
    assert checkpoint_manifest["adapted_dataset"]["test_jsonl_sha256"] == file_sha256(TEST_PATH)
    assert checkpoint_manifest["adapter_model"]["sha256"] == file_sha256(
        ADAPTER_DIR / "adapter_model.safetensors"
    )
    ADAPTER_SOURCE = f"{STAGING_REPO_ID}@{STAGING_REVISION}:{STAGING_ADAPTER_PATH}"
else:
    ADAPTER_DIR = Path(snapshot_download(MODEL_REPO_ID, token=HF_TOKEN))
    ADAPTER_SOURCE = MODEL_REPO_ID

adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
checkpoint_base = adapter_config["base_model_name_or_path"]
checkpoint_slug = checkpoint_base.lower().replace("reference", "").replace("__tog__ft", "")
expected_slug = EXPECTED_BASE_MODEL_ID.lower().split("/")[-1]
assert expected_slug in checkpoint_slug or checkpoint_slug.endswith(expected_slug), (
    f"checkpoint base {checkpoint_base!r} does not match {EXPECTED_BASE_MODEL_ID!r}"
)
BASE_MODEL_ID = EXPECTED_BASE_MODEL_ID
USE_LIVE_INFERENCE = True
BASE_PREDICTION_SOURCE = Path()
ADAPTED_PREDICTION_SOURCE = Path()
print("adapter:", ADAPTER_SOURCE)
print("base model:", BASE_MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model_kwargs = {
    "token": HF_TOKEN,
    "torch_dtype": (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16 if torch.cuda.is_available() else torch.float32
    ),
    "device_map": "auto",
    "low_cpu_mem_usage": True,
}
USE_4BIT = os.environ.get("FALSIFYRL_USE_4BIT", "false").lower() == "true"
if USE_4BIT:
    assert torch.cuda.is_available(), (
        "4-bit inference requires a Colab GPU runtime; select L4 or A100 before rerunning"
    )
    try:
        import bitsandbytes  # noqa: F401
    except ImportError as error:
        raise RuntimeError(
            "4-bit inference was requested but bitsandbytes is unavailable; rerun the install cell"
        ) from error
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=(
            torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        ),
    )
try:
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
except (TypeError, ValueError):
    base_model = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
base_model.eval()


In [ ]:
PATCH_FIELD_ALIASES = {
    "idle_weight": "idle_agent_weight",
    "completion_weight": "completion_bonus",
}
FAILURE_TYPE_ALIASES = {
    "idle_waste": "no_op_bonus",
    "idle_wait": "no_op_bonus",
}
OUTPUT_CANONICALIZER = "falsifyrl_schema_aliases_v1"

def canonicalize_schema_aliases(value):
    failure_type = value.get("failure_type")
    if failure_type in FAILURE_TYPE_ALIASES:
        value["failure_type"] = FAILURE_TYPE_ALIASES[failure_type]
    patch = value.get("reward_patch")
    if isinstance(patch, dict) and isinstance(patch.get("updates"), dict):
        updates = patch["updates"]
        normalized = {
            PATCH_FIELD_ALIASES.get(field, field): update
            for field, update in updates.items()
        }
        if len(normalized) == len(updates):
            patch["updates"] = normalized
    return value

def extract_json(text):
    decoder = json.JSONDecoder()
    candidates = []
    for index, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text, index)
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            candidates.append(value)
    preferred = [
        value for value in candidates
        if {"verdict", "failure_type"}.issubset(value)
    ]
    if preferred or candidates:
        return json.dumps(
            canonicalize_schema_aliases((preferred or candidates)[-1]),
            separators=(",", ":"),
            sort_keys=True,
        )
    return text.strip()

def predict(model, prompts, batch_size):
    predictions = []
    for start in range(0, len(prompts), batch_size):
        prompt_batch = prompts[start:start + batch_size]
        formatted = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for prompt in prompt_batch
        ]
        inputs = tokenizer(
            formatted,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=int(os.environ.get("FALSIFYRL_MAX_NEW_TOKENS", 768)),
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.batch_decode(
            outputs[:, inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )
        predictions.extend(extract_json(text) for text in generated)
        print(f"generated {len(predictions)}/{len(prompts)}")
    return predictions

def load_predictions(path):
    values = {
        item["example_id"]: item["completion"]
        for item in (
            json.loads(line) for line in path.read_text().splitlines() if line.strip()
        )
    }
    expected_ids = {row["example_id"] for row in rows}
    assert set(values) == expected_ids
    return [values[row["example_id"]] for row in rows]

MAX_EXAMPLES = int(os.environ.get("FALSIFYRL_MAX_EXAMPLES", len(rows)))
BATCH_SIZE = int(os.environ.get("FALSIFYRL_BATCH_SIZE", 1))
prompts = [row["prompt"] for row in rows[:MAX_EXAMPLES]]
if USE_LIVE_INFERENCE:
    base_predictions = predict(base_model, prompts, BATCH_SIZE)
else:
    assert len(rows) == MAX_EXAMPLES, "cached evidence is always the exact 640-case split"
    base_predictions = load_predictions(BASE_PREDICTION_SOURCE)
base_metrics = compact_metrics(base_predictions)
base_metrics


In [ ]:
if USE_LIVE_INFERENCE:
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model.eval()
    adapted_predictions = predict(model, prompts, BATCH_SIZE)
else:
    adapted_predictions = load_predictions(ADAPTED_PREDICTION_SOURCE)
adapted_metrics = compact_metrics(adapted_predictions)
{
    "base": base_metrics,
    "adapted": adapted_metrics,
    "verdict_macro_f1_improvement": (
        adapted_metrics["verdict_macro_f1"] - base_metrics["verdict_macro_f1"]
    ),
}


In [ ]:
def save_predictions(path, predictions):
    with path.open("w") as stream:
        for row, completion in zip(
            rows[:MAX_EXAMPLES], predictions, strict=True
        ):
            stream.write(json.dumps({
                "example_id": row["example_id"],
                "completion": completion,
            }) + "\n")

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

COLAB_OUTPUT_DIR = Path("/content/FalsifyRL/evaluation") / RUN_ID
COLAB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_prediction_path = COLAB_OUTPUT_DIR / "falsifyrl-base-test-predictions.jsonl"
adapted_prediction_path = COLAB_OUTPUT_DIR / "falsifyrl-adapted-test-predictions.jsonl"
save_predictions(base_prediction_path, base_predictions)
save_predictions(adapted_prediction_path, adapted_predictions)

adapter_weights = ADAPTER_DIR / "adapter_model.safetensors"
report = {
    "run_id": RUN_ID,
    "dataset_test_path": str(TEST_PATH),
    "adapter_path": str(ADAPTER_DIR),
    "adapter_sha256": sha256(adapter_weights),
    "base_model_id": BASE_MODEL_ID,
    "example_count": MAX_EXAMPLES,
    "base_predictions_sha256": sha256(base_prediction_path),
    "adapted_predictions_sha256": sha256(adapted_prediction_path),
    "output_canonicalizer": {
        "name": OUTPUT_CANONICALIZER,
        "patch_field_aliases": PATCH_FIELD_ALIASES,
        "failure_type_aliases": FAILURE_TYPE_ALIASES,
    },
    "base_metrics": base_metrics,
    "adapted_metrics": adapted_metrics,
    "improvement": {
        key: adapted_metrics[key] - base_metrics[key]
        for key in adapted_metrics
        if isinstance(adapted_metrics[key], float)
    },
}
report_path = COLAB_OUTPUT_DIR / "colab-evaluation.json"
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
artifact_manifest = {
    "schema_version": 1,
    "autoscientist_run_id": RUN_ID,
    "base_model_id": BASE_MODEL_ID,
    "checkpoint_revision": STAGING_REVISION or None,
    "adapter_sha256": report["adapter_sha256"],
    "test_jsonl_sha256": file_sha256(TEST_PATH),
    "example_count": MAX_EXAMPLES,
    "batch_size": BATCH_SIZE,
    "max_new_tokens": int(os.environ.get("FALSIFYRL_MAX_NEW_TOKENS", 768)),
    "do_sample": False,
    "output_canonicalizer": report["output_canonicalizer"],
    "files": {
        base_prediction_path.name: {
            "sha256": report["base_predictions_sha256"],
            "bytes": base_prediction_path.stat().st_size,
        },
        adapted_prediction_path.name: {
            "sha256": report["adapted_predictions_sha256"],
            "bytes": adapted_prediction_path.stat().st_size,
        },
        report_path.name: {
            "sha256": sha256(report_path),
            "bytes": report_path.stat().st_size,
        },
    },
}
artifact_manifest_path = COLAB_OUTPUT_DIR / "evaluation-manifest.json"
artifact_manifest_path.write_text(
    json.dumps(artifact_manifest, indent=2, sort_keys=True) + "\n"
)
if STAGING_REPO_ID:
    from huggingface_hub import CommitOperationAdd, HfApi

    api = HfApi(token=HF_TOKEN)
    current_head = api.repo_info(
        repo_id=STAGING_REPO_ID,
        repo_type="model",
    ).sha
    evidence_path = f"runs/{RUN_ID}/evaluation"
    operations = [
        CommitOperationAdd(
            path_in_repo=f"{evidence_path}/{path.name}",
            path_or_fileobj=str(path),
        )
        for path in (
            base_prediction_path,
            adapted_prediction_path,
            report_path,
            artifact_manifest_path,
        )
    ]
    commit = api.create_commit(
        repo_id=STAGING_REPO_ID,
        repo_type="model",
        operations=operations,
        commit_message=f"Upload private Colab evaluation evidence for {RUN_ID}",
        parent_commit=current_head,
    )
    print("private evaluation revision:", commit.oid)
print(json.dumps(report, indent=2, sort_keys=True))
print("saved:", COLAB_OUTPUT_DIR)


When private staging is configured, the notebook uploads both prediction
JSONL files and their hash manifest to the private repository. Otherwise download them from the
Colab file browser. The repository's CPU-only evaluator validates the complete output schema and
re-executes every proposed reward patch.
This Colab notebook performs only the GPU-heavy base and adapter inference over the exact same 640
held-out examples.
